In [1]:
from google.colab import files
uploaded = files.upload()

Saving customer_churn.csv to customer_churn.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

In [3]:
df = pd.read_csv("customer_churn.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn Names:")
print(df.columns.tolist())

Shape: (7043, 21)

First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Str

In [4]:
np.random.seed(42)

# Add the 2 missing columns
df["Age"] = np.random.randint(18, 71, size=len(df))
df["SupportTickets"] = np.random.randint(0, 11, size=len(df))


df = df.rename(columns={
    "customerID": "CustomerID",
    "gender": "Gender",
    "tenure": "Tenure",
    "Contract": "ContractType"
})

# Keep only the 11 columns the assignment requires
df = df[[
    "CustomerID",
    "Gender",
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ContractType",
    "PaymentMethod",
    "InternetService",
    "SupportTickets",
    "Churn"
]]

print("Shape after preparation:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

Shape after preparation: (7043, 11)

Column Names:
['CustomerID', 'Gender', 'Age', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'ContractType', 'PaymentMethod', 'InternetService', 'SupportTickets', 'Churn']

First 5 rows:
   CustomerID  Gender  Age  Tenure  MonthlyCharges TotalCharges  \
0  7590-VHVEG  Female   56       1           29.85        29.85   
1  5575-GNVDE    Male   69      34           56.95       1889.5   
2  3668-QPYBK    Male   46       2           53.85       108.15   
3  7795-CFOCW    Male   32      45           42.30      1840.75   
4  9237-HQITU  Female   60       2           70.70       151.65   

     ContractType              PaymentMethod InternetService  SupportTickets  \
0  Month-to-month           Electronic check             DSL               2   
1        One year               Mailed check             DSL              10   
2  Month-to-month               Mailed check             DSL              10   
3        One year  Bank transfer (automatic)            

In [5]:
print("Basic Info")
print(df.info())

print("\n Missing Values")
print(df.isnull().sum())

print("\n Duplicate Rows")
print(df.duplicated().sum())

print("\n Target Column Distribution")
print(df["Churn"].value_counts())

print("\nChurn percentage:")
print(df["Churn"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print("\n Numerical Columns Summary")
print(df[["Age", "Tenure", "MonthlyCharges", "TotalCharges", "SupportTickets"]].describe())

Basic Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       7043 non-null   object 
 1   Gender           7043 non-null   object 
 2   Age              7043 non-null   int64  
 3   Tenure           7043 non-null   int64  
 4   MonthlyCharges   7043 non-null   float64
 5   TotalCharges     7043 non-null   object 
 6   ContractType     7043 non-null   object 
 7   PaymentMethod    7043 non-null   object 
 8   InternetService  7043 non-null   object 
 9   SupportTickets   7043 non-null   int64  
 10  Churn            7043 non-null   object 
dtypes: float64(1), int64(3), object(7)
memory usage: 605.4+ KB
None

 Missing Values
CustomerID         0
Gender             0
Age                0
Tenure             0
MonthlyCharges     0
TotalCharges       0
ContractType       0
PaymentMethod      0
InternetService    0
SupportTi

In [6]:
# TotalCharges is stored as text — convert it to numeric
# Blank spaces will become NaN (missing values) automatically
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("TotalCharges dtype after fix:", df["TotalCharges"].dtype)
print("\nMissing values after fix:")
print(df.isnull().sum())
print("\nTotalCharges summary:")
print(df["TotalCharges"].describe())

TotalCharges dtype after fix: float64

Missing values after fix:
CustomerID          0
Gender              0
Age                 0
Tenure              0
MonthlyCharges      0
TotalCharges       11
ContractType        0
PaymentMethod       0
InternetService     0
SupportTickets      0
Churn               0
dtype: int64

TotalCharges summary:
count    7032.000000
mean     2283.300441
std      2266.771362
min        18.800000
25%       401.450000
50%      1397.475000
75%      3794.737500
max      8684.800000
Name: TotalCharges, dtype: float64


In [7]:
df = df.drop("CustomerID", axis=1)

print("Shape after dropping CustomerID:", df.shape)
print("Columns remaining:", df.columns.tolist())

Shape after dropping CustomerID: (7043, 10)
Columns remaining: ['Gender', 'Age', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'ContractType', 'PaymentMethod', 'InternetService', 'SupportTickets', 'Churn']


In [8]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
numeric_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SupportTickets"
]

categorical_features = [
    "Gender",
    "ContractType",
    "PaymentMethod",
    "InternetService"
]

In [11]:


print("\nNumerical columns sample:")
print(X_train[numeric_features].head())

print("\nCategorical columns sample:")
print(X_train[categorical_features].head())


Numerical columns sample:
      Age  Tenure  MonthlyCharges  TotalCharges  SupportTickets
3738   62      35           49.20       1701.65               5
3151   70      15           75.10       1151.55               4
4860   42      13           40.55        590.35               6
3867   68      26           73.50       1905.70              10
3810   63       1           44.55         44.55               3

Categorical columns sample:
      Gender    ContractType            PaymentMethod InternetService
3738    Male  Month-to-month         Electronic check             DSL
3151    Male  Month-to-month             Mailed check     Fiber optic
4860    Male        Two year             Mailed check             DSL
3867  Female        Two year  Credit card (automatic)             DSL
3810    Male  Month-to-month         Electronic check             DSL


In [12]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [13]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [15]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

In [16]:
model_pipeline.fit(X_train, y_train)

print("Pipeline training complete.")

Pipeline training complete.


In [17]:
y_pred = model_pipeline.predict(X_test)

print("Predictions complete.")
print("Total predictions made:", len(y_pred))
print("\nSample predictions (first 10):")
print(y_pred[:10])
print("\nActual values (first 10):")
print(y_test.values[:10])

Predictions complete.
Total predictions made: 1409

Sample predictions (first 10):
['No' 'Yes' 'No' 'Yes' 'No' 'No' 'Yes' 'No' 'No' 'No']

Actual values (first 10):
['No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'Yes']


In [18]:
print("\n1. Accuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\n2. Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n3. Classification Report:")
print(classification_report(y_test, y_pred))


1. Accuracy Score:
0.7849538679914834

2. Confusion Matrix:
[[920 115]
 [188 186]]

3. Classification Report:
              precision    recall  f1-score   support

          No       0.83      0.89      0.86      1035
         Yes       0.62      0.50      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.78      1409



In [19]:
joblib.dump(model_pipeline, "customer_churn_pipeline.pkl")

print("Pipeline saved successfully")

Pipeline saved successfully


In [20]:
loaded_pipeline = joblib.load("customer_churn_pipeline.pkl")

print("Pipeline loaded successfully.")

Pipeline loaded successfully.


In [21]:
new_customer = pd.DataFrame({
    "Gender": ["Female"],
    "Age": [32],
    "Tenure": [5],
    "MonthlyCharges": [850],
    "TotalCharges": [4250],
    "ContractType": ["Month-to-month"],
    "PaymentMethod": ["Electronic check"],
    "InternetService": ["Fiber optic"],
    "SupportTickets": [3]
})

prediction = loaded_pipeline.predict(new_customer)
prediction_proba = loaded_pipeline.predict_proba(new_customer)

print(prediction)


print("\nBusiness Interpretation")
if prediction[0] == "Yes":
    print("This customer is LIKELY TO CHURN.")
    print("Recommended actions:")
    print("  - Offer a discount or loyalty reward")
    print("  - Suggest switching to a longer contract")
    print("  - Assign a dedicated support representative")
else:
    print("This customer is NOT likely to churn.")
    print("Recommended actions:")
    print("  - Continue regular engagement")
    print("  - Monitor monthly for any changes")

['Yes']

Business Interpretation
This customer is LIKELY TO CHURN.
Recommended actions:
  - Offer a discount or loyalty reward
  - Suggest switching to a longer contract
  - Assign a dedicated support representative
